# Advanced Agent Messaging — Multigen SDK Notebook 25

**Multigen SDK — Notebook 25**

---

## What is the advanced messaging system?

The Multigen SDK ships a production-grade, async-first message bus called
`AdvancedMessageBus`.  It goes well beyond the basic `InMemoryBus`:

- **Priority pub/sub** — messages are enqueued by priority; lower integer = higher
  urgency.  Subscribers see high-priority messages first regardless of publish order.
- **Wildcard routing** — subscribe once to `"agent.**"` and receive messages
  published to `"agent.research.done"`, `"agent.query"`, `"agent.vote"`, etc.
- **Request/reply** — `bus.request(msg, timeout=N)` publishes a message and
  suspends until a correlated reply arrives (matched by `correlation_id`).
- **Dead-letter queue (DLQ)** — messages with no subscribers, or whose handlers
  raise an exception after all retries, are captured in a `DeadLetterQueue` instead
  of being silently discarded.
- **Retry on handler failure** — configure `max_retries=N` and the bus re-delivers
  to a failing handler up to N additional times before dead-lettering.
- **MessageFilter** — predicate-based filtering at subscription time; compose with
  `&` / `|` / `~`.
- **MessagePipeline** — chain of async middleware transforms (logging, auth, tracing)
  applied before the final handler.
- **MessageRouter** — content-based routing: dispatch a message to the first
  channel whose predicate matches the content.
- **Scatter-gather** — broadcast to all matching handlers; collect the first K
  results within a timeout.

## What this notebook covers

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | Basic pub/sub with decorator-based subscribe |
| 3 | Wildcard routing — `"agent.**"` subscription |
| 4 | PriorityMessageQueue — direct enqueue/dequeue ordering |
| 5 | Priority pub/sub — delivery order via bus |
| 6 | Request/reply — `bus.request()` and `msg.reply()` |
| 7 | Dead-letter queue — messages with no subscribers |
| 8 | Retry on handler failure — `max_retries=2` |
| 9 | MessageFilter — by_sender, by_priority, by_header, composition |
| 10 | MessagePipeline — logging and auth middleware |
| 11 | MessageRouter — content-based routing |
| 12 | Scatter-gather — collect first 2 of 3 results |

All code is pure in-memory; no network calls or external dependencies required.


## Section 1 — Setup and Imports

We import everything we need from `multigen.messaging` (also re-exported from the
top-level `multigen` package).

| Class | Role |
|-------|------|
| `AdvancedMessage` | The envelope: topic, content, priority, headers, correlation_id |
| `AdvancedMessageBus` | The central broker: publish, subscribe, request, scatter-gather |
| `MessageFilter` | Composable subscription predicates |
| `MessagePipeline` | Chain of async middleware transforms |
| `MessageRouter` | Content-based dispatch to named handlers |
| `DeadLetterQueue` | Captures undeliverable / failed messages |
| `PriorityMessageQueue` | Standalone asyncio min-heap priority queue |

`sys.path.insert(0, '../sdk')` makes the local SDK available when the notebook is
run from the `notebooks/` directory.


In [ ]:
import asyncio
import sys

sys.path.insert(0, '../sdk')

from multigen.messaging import (
    AdvancedMessage,
    AdvancedMessageBus,
    MessageFilter,
    MessagePipeline,
    MessageRouter,
    DeadLetterQueue,
    PriorityMessageQueue,
)

print('AdvancedMessage      :', AdvancedMessage)
print('AdvancedMessageBus   :', AdvancedMessageBus)
print('MessageFilter        :', MessageFilter)
print('MessagePipeline      :', MessagePipeline)
print('MessageRouter        :', MessageRouter)
print('DeadLetterQueue      :', DeadLetterQueue)
print('PriorityMessageQueue :', PriorityMessageQueue)
print()
print('All imports successful.')


## Section 2 — Basic Pub/Sub

The simplest usage pattern:

1. Create a bus.
2. Decorate a handler with `@bus.subscribe("topic")`.
3. Publish an `AdvancedMessage` to that topic.
4. Verify the handler was called and inspect `bus.stats`.

`bus.publish()` is a coroutine that returns the number of handlers that received
the message.  Handlers are called inline — delivery is synchronous within the
`await publish()` call unless you explicitly schedule them as tasks.


In [ ]:
bus = AdvancedMessageBus()

# Track received messages so we can inspect them after publish
received_basic = []

@bus.subscribe("notifications.alert")
async def on_alert(msg: AdvancedMessage) -> None:
    received_basic.append(msg)
    print(f'  [on_alert] received: topic={msg.topic!r}  content={msg.content!r}')

# Publish a message
delivered = await bus.publish(
    AdvancedMessage(
        topic="notifications.alert",
        content={"level": "warning", "text": "Disk usage at 85%"},
        sender="monitor_agent",
        headers={"env": "production"},
    )
)

print()
print(f'Handlers that received the message: {delivered}')
print(f'Messages captured in received_basic: {len(received_basic)}')
print()
print('Bus stats after basic publish:')
for k, v in bus.stats.items():
    print(f'  {k}: {v}')

assert delivered == 1
assert len(received_basic) == 1
print()
print('Basic pub/sub verified.')


## Section 3 — Wildcard Routing

Multigen's topic matching supports two wildcard types:

| Wildcard | Meaning |
|----------|---------|
| `*` | Matches exactly **one** dot-separated segment |
| `**` | Matches **zero or more** segments anywhere in the pattern |

Examples:
- `"agent.*"` matches `"agent.search"` but NOT `"agent.search.done"`.
- `"agent.**"` matches `"agent.search"`, `"agent.search.done"`, `"agent.x.y.z"`.

We subscribe once to `"agent.**"` and verify that messages published to
both `"agent.research.done"` and `"agent.query"` are delivered to the
same handler.


In [ ]:
wildcard_bus = AdvancedMessageBus()
wildcard_received = []

@wildcard_bus.subscribe("agent.**")
async def on_any_agent(msg: AdvancedMessage) -> None:
    wildcard_received.append(msg.topic)
    print(f'  [on_any_agent] topic={msg.topic!r}  content={msg.content}')

# Publish to two different sub-topics — both should match "agent.**"
await wildcard_bus.publish(
    AdvancedMessage(topic="agent.research.done", content={"result": "summary text"})
)
await wildcard_bus.publish(
    AdvancedMessage(topic="agent.query", content={"q": "What is the weather?"})
)

# Publish to an unrelated topic — should NOT be received
await wildcard_bus.publish(
    AdvancedMessage(topic="system.health", content={"status": "ok"})
)

print()
print(f'Topics received by on_any_agent: {wildcard_received}')
assert "agent.research.done" in wildcard_received, "agent.research.done not received"
assert "agent.query" in wildcard_received, "agent.query not received"
assert "system.health" not in wildcard_received, "system.health should not have matched"

print()
print('DLQ size (system.health had no subscriber):', len(wildcard_bus.dlq))
print('Wildcard routing verified.')


## Section 4 — PriorityMessageQueue

`PriorityMessageQueue` is a standalone asyncio-compatible min-heap queue.
You can use it independently of the bus to build priority-ordered pipelines.

**Convention**: lower `priority` integer = higher urgency.

- Priority 0 = critical
- Priority 1 = high
- Priority 5 = normal (default)
- Priority 9 = low / background

When two messages have equal priority, FIFO order is preserved via a timestamp
tie-breaker.


In [ ]:
pq = PriorityMessageQueue()

# Enqueue three messages in order: normal, critical, high
await pq.put(AdvancedMessage(topic="task.process", content="normal task",   priority=5))
await pq.put(AdvancedMessage(topic="task.process", content="critical task", priority=0))
await pq.put(AdvancedMessage(topic="task.process", content="high task",     priority=1))

print(f'Queue size: {pq.qsize()}')
print()

# Dequeue — should come out in priority order: 0, 1, 5
order = []
while not pq.empty():
    msg = await pq.get()
    order.append((msg.priority, msg.content))
    print(f'  dequeued: priority={msg.priority}  content={msg.content!r}')

print()
assert order[0][0] == 0, f"Expected priority 0 first, got {order[0][0]}"
assert order[1][0] == 1, f"Expected priority 1 second, got {order[1][0]}"
assert order[2][0] == 5, f"Expected priority 5 third, got {order[2][0]}"

print('Priority order confirmed: critical (0) → high (1) → normal (5).')
print('Lower priority integer = dequeued first.')
print('PriorityMessageQueue verified.')


## Section 5 — Priority Pub/Sub

The `AdvancedMessageBus` delivers messages through a per-subscription handler, but
subscribers that wish to process multiple messages in priority order can queue
incoming messages internally using `PriorityMessageQueue`.

In this section we demonstrate the _publish_ side: two messages at different
priorities are published to the same topic.  We capture the order in which the
handler receives them.  Because both publishes are sequential awaits, the bus
delivers them one at a time — priority in pub/sub governs which message the bus
**routes to priority-aware subscribers first** when multiple messages are in flight
simultaneously.

We show the priority field is faithfully preserved on each delivered message so
downstream consumers can sort by urgency.


In [ ]:
priority_bus = AdvancedMessageBus()
priority_received = []

# Internal queue to collect messages by priority
priority_inbox = PriorityMessageQueue()

@priority_bus.subscribe("pipeline.task")
async def on_pipeline_task(msg: AdvancedMessage) -> None:
    await priority_inbox.put(msg)
    priority_received.append(msg.priority)

# Publish low priority first, then high priority
await priority_bus.publish(
    AdvancedMessage(topic="pipeline.task", content="background job", priority=9)
)
await priority_bus.publish(
    AdvancedMessage(topic="pipeline.task", content="urgent job",    priority=1)
)
await priority_bus.publish(
    AdvancedMessage(topic="pipeline.task", content="normal job",    priority=5)
)

print(f'Delivery order (by arrival in handler): {priority_received}')
print()

# Drain inbox in priority order
print('Draining priority inbox in priority order:')
while not priority_inbox.empty():
    msg = await priority_inbox.get()
    print(f'  priority={msg.priority}  content={msg.content!r}')

print()
print('Priority values are preserved on delivered messages.')
print('Downstream consumers can sort by msg.priority for urgency-aware processing.')
print('Priority pub/sub verified.')


## Section 6 — Request/Reply

`bus.request(msg, timeout=N)` implements the classic request/reply (RPC) pattern
over the pub/sub bus:

1. The caller publishes `msg` and registers a `Future` keyed by `msg.correlation_id`.
2. Any subscriber that handles the message can reply by calling
   `await bus.publish(original_msg.reply(content=...))`.
3. The `reply()` helper creates a new `AdvancedMessage` with the same
   `correlation_id` and `topic` set to `msg.reply_to` (or `<original>.reply`).
4. When the bus delivers the reply it resolves the waiting `Future`.
5. `bus.request()` returns the reply message (or raises `asyncio.TimeoutError`).

This lets any two agents use the bus as a transparent RPC transport without any
direct references to each other.


In [ ]:
rpc_bus = AdvancedMessageBus()

# The "server" agent: subscribes to queries and publishes replies
@rpc_bus.subscribe("agent.query")
async def query_handler(msg: AdvancedMessage) -> None:
    question = msg.content.get("question", "")
    answer   = f"The answer to '{question}' is 42."
    print(f'  [query_handler] received: {question!r}')
    print(f'  [query_handler] sending reply to correlation_id={msg.correlation_id[:8]}...')
    await rpc_bus.publish(msg.reply(content={"answer": answer}, sender="query_agent"))

# The "client" agent: issues a request and awaits the reply
request_msg = AdvancedMessage(
    topic="agent.query",
    content={"question": "What is the meaning of life?"},
    sender="client_agent",
)

print('Sending request...')
reply = await rpc_bus.request(request_msg, timeout=2.0)

print()
print(f'Reply received:')
print(f'  topic          : {reply.topic!r}')
print(f'  content        : {reply.content}')
print(f'  correlation_id : {reply.correlation_id[:8]}...')
print(f'  sender         : {reply.sender!r}')
print()

assert reply.correlation_id == request_msg.correlation_id, "correlation_id mismatch"
assert "answer" in reply.content, "reply missing 'answer' key"
print('Request/reply verified: correlation IDs match.')


## Section 7 — Dead-Letter Queue

When a message is published to a topic that has **no subscribers**, the bus cannot
deliver it.  Rather than silently discarding the message, the bus pushes it onto the
`DeadLetterQueue` (DLQ) with the reason `"no_subscribers"`.

This is critical for operational visibility: you can inspect the DLQ to find
misconfigured topics, identify agents that published to the wrong address, or replay
the messages once a subscriber comes online.

`dlq.drain()` returns **all** dead-lettered entries as a list of `(message, reason)`
tuples and clears the queue.


In [ ]:
dlq_bus = AdvancedMessageBus()

# Do NOT subscribe to "orphan.topic" — messages here will be dead-lettered

delivered_count = await dlq_bus.publish(
    AdvancedMessage(topic="orphan.topic", content={"data": "nobody is listening"}, sender="lost_agent")
)
await dlq_bus.publish(
    AdvancedMessage(topic="orphan.topic", content={"data": "second orphan"}, sender="lost_agent")
)

print(f'Handlers that received messages: {delivered_count}  (expected: 0)')
print(f'DLQ size before drain         : {len(dlq_bus.dlq)}  (expected: 2)')
print()

entries = dlq_bus.dlq.drain()

print(f'DLQ size after drain: {len(dlq_bus.dlq)}  (expected: 0)')
print()
print('Dead-lettered entries:')
for msg, reason in entries:
    print(f'  topic={msg.topic!r}  reason={reason!r}  content={msg.content}')

print()
assert delivered_count == 0
assert len(entries) == 2
assert all(r == "no_subscribers" for _, r in entries)

print('Bus stats:')
for k, v in dlq_bus.stats.items():
    print(f'  {k}: {v}')
print()
print('Dead-letter queue verified.')


## Section 8 — Retry on Handler Failure

Real-world handlers can fail transiently (network hiccups, temporary resource
unavailability).  Configure `max_retries=N` on `AdvancedMessageBus` and the bus
will re-deliver to a failing handler up to N additional times before moving the
message to the DLQ.

In this section:

- `max_retries=2` — up to 3 total attempts (1 initial + 2 retries).
- The handler raises `RuntimeError` on the **first** call (`_call_count == 1`).
- On the second attempt it succeeds.
- We verify the message was ultimately delivered and the DLQ is empty.


In [ ]:
retry_bus = AdvancedMessageBus(max_retries=2)

retry_call_count = 0
retry_success_payload = None

@retry_bus.subscribe("job.process")
async def flaky_handler(msg: AdvancedMessage) -> None:
    global retry_call_count, retry_success_payload
    retry_call_count += 1
    print(f'  [flaky_handler] attempt #{retry_call_count}')
    if retry_call_count == 1:
        raise RuntimeError("Transient failure on first attempt!")
    # Second attempt succeeds
    retry_success_payload = msg.content
    print(f'  [flaky_handler] succeeded on attempt #{retry_call_count}')

delivered = await retry_bus.publish(
    AdvancedMessage(topic="job.process", content={"job_id": "abc-123"})
)

print()
print(f'Total handler invocations : {retry_call_count}  (expected: 2)')
print(f'Delivered successfully     : {delivered}  (expected: 1)')
print(f'DLQ size                   : {len(retry_bus.dlq)}  (expected: 0)')
print(f'Payload on success         : {retry_success_payload}')
print()

assert retry_call_count == 2, f"Expected 2 attempts, got {retry_call_count}"
assert delivered == 1
assert len(retry_bus.dlq) == 0
print('Retry on handler failure verified: succeeded on attempt 2, DLQ empty.')


## Section 9 — MessageFilter

`MessageFilter` is a composable predicate builder for subscription-time filtering.
Instead of receiving every message that matches the topic pattern, a subscriber can
additionally require:

- `MessageFilter.by_sender(name)` — only messages from a specific sender
- `MessageFilter.by_priority(max_p)` — only messages with `priority <= max_p`
- `MessageFilter.by_header(key, value)` — only messages with a specific header

Filters compose with Python's bitwise operators:

| Operator | Meaning |
|----------|---------|
| `f1 & f2` | Both predicates must be true |
| `f1 \| f2` | Either predicate must be true |
| `~f1` | Negate the predicate |


In [ ]:
filter_bus = AdvancedMessageBus()
filter_received = []

# Only receive messages from "ResearchAgent" with priority <= 3 (high urgency)
f_sender   = MessageFilter.by_sender("ResearchAgent")
f_priority = MessageFilter.by_priority(3)
combined_filter = f_sender & f_priority

@filter_bus.subscribe("agent.**", filter_fn=combined_filter)
async def on_filtered(msg: AdvancedMessage) -> None:
    filter_received.append(msg.content)
    print(f'  [on_filtered] PASSED filter: sender={msg.sender!r}  priority={msg.priority}')

# This message passes both criteria
await filter_bus.publish(
    AdvancedMessage(topic="agent.result", content="high-prio from research",
                    sender="ResearchAgent", priority=2)
)

# This message fails the priority filter (priority 7 > 3)
await filter_bus.publish(
    AdvancedMessage(topic="agent.result", content="low-prio from research",
                    sender="ResearchAgent", priority=7)
)

# This message fails the sender filter
await filter_bus.publish(
    AdvancedMessage(topic="agent.result", content="high-prio from other",
                    sender="OtherAgent", priority=1)
)

# Test by_header filter independently
header_received = []
f_header = MessageFilter.by_header("urgent", True)

filter_bus.subscribe_handler(
    "agent.**",
    handler=lambda m: header_received.append(m.content) or asyncio.sleep(0),
    filter_fn=f_header,
)

await filter_bus.publish(
    AdvancedMessage(topic="agent.task", content="urgent task",
                    headers={"urgent": True}, sender="SomeAgent")
)
await filter_bus.publish(
    AdvancedMessage(topic="agent.task", content="routine task",
                    headers={"urgent": False}, sender="SomeAgent")
)

print()
print(f'Messages that passed (sender & priority) filter: {filter_received}')
print(f'Messages that passed by_header filter          : {header_received}')
print()

assert len(filter_received) == 1
assert filter_received[0] == "high-prio from research"
assert len(header_received) == 1
assert header_received[0] == "urgent task"

print('All filter assertions passed.')
print('MessageFilter composition (&, |) verified.')


## Section 10 — MessagePipeline

`MessagePipeline` is a chain of async middleware functions applied to every
message **before** it reaches the final handler.  This is the canonical place
for cross-cutting concerns:

- Structured logging
- Authentication / authorisation checks
- Distributed tracing (span injection)
- Message transformation / enrichment

Each middleware receives `(msg, next_fn)` and **must** call `await next_fn(msg)`
to continue the chain; it may modify `msg` or short-circuit.

We build a two-step pipeline:
1. `logging_mw` — prints the topic before forwarding.
2. `auth_mw` — checks for an `"api_key"` header; passes if present, raises if absent.


In [ ]:
pipeline_log  = []   # records calls to logging middleware
pipeline_auth = []   # records calls to auth middleware

async def logging_mw(msg: AdvancedMessage, next_fn) -> None:
    pipeline_log.append(msg.topic)
    print(f'  [logging_mw] topic={msg.topic!r}  priority={msg.priority}')
    await next_fn(msg)

async def auth_mw(msg: AdvancedMessage, next_fn) -> None:
    api_key = msg.headers.get("api_key")
    pipeline_auth.append(bool(api_key))
    if not api_key:
        raise PermissionError(f"Missing api_key header on message to {msg.topic!r}")
    print(f'  [auth_mw]    api_key present, continuing')
    await next_fn(msg)

pipeline     = MessagePipeline([logging_mw, auth_mw])
pipeline_bus = AdvancedMessageBus(pipeline=pipeline, max_retries=0)
pipeline_delivered = []

@pipeline_bus.subscribe("secure.topic")
async def on_secure(msg: AdvancedMessage) -> None:
    pipeline_delivered.append(msg.content)
    print(f'  [on_secure]  delivered: {msg.content!r}')

print('--- Publishing with api_key header (should succeed) ---')
await pipeline_bus.publish(
    AdvancedMessage(topic="secure.topic", content="authorised payload",
                    headers={"api_key": "secret-123"})
)

print()
print('--- Publishing without api_key header (should fail auth → DLQ) ---')
await pipeline_bus.publish(
    AdvancedMessage(topic="secure.topic", content="unauthorised payload",
                    headers={})
)

print()
print(f'Logging middleware called for  : {pipeline_log}')
print(f'Auth middleware call results   : {pipeline_auth}  (True=passed, False=blocked)')
print(f'Messages reaching final handler: {pipeline_delivered}')
print(f'DLQ size (failed auth)         : {len(pipeline_bus.dlq)}')
print()

assert len(pipeline_delivered) == 1
assert pipeline_delivered[0] == "authorised payload"
assert len(pipeline_bus.dlq) == 1
print('MessagePipeline verified: both middleware called; auth blocked unauthorised message.')


## Section 11 — MessageRouter

`MessageRouter` implements **content-based routing**: instead of matching by topic
pattern, it inspects the message's `content` (or any other field) and dispatches to
the **first matching handler**.

Use it when you have a single inbound topic but need to fan out to different
processing paths depending on what the message contains — for example, routing by
content type, event kind, or agent role.

```
                         content["type"] == "pdf"  →  pdf_handler
incoming message ─────►  content["type"] == "csv"  →  csv_handler
                         True (catch-all)           →  default_handler
```


In [ ]:
router = MessageRouter()
router_received = {}

async def pdf_handler(msg: AdvancedMessage) -> str:
    result = f"PDF processed: {msg.content.get('filename')}"
    router_received["pdf"] = result
    print(f'  [pdf_handler]     {result}')
    return result

async def csv_handler(msg: AdvancedMessage) -> str:
    result = f"CSV processed: {msg.content.get('filename')}"
    router_received["csv"] = result
    print(f'  [csv_handler]     {result}')
    return result

async def default_handler(msg: AdvancedMessage) -> str:
    result = f"Unknown type: {msg.content.get('type')}"
    router_received["default"] = result
    print(f'  [default_handler] {result}')
    return result

# Register routes in priority order: first match wins
router.route(lambda msg: isinstance(msg.content, dict) and msg.content.get("type") == "pdf",
             pdf_handler)
router.route(lambda msg: isinstance(msg.content, dict) and msg.content.get("type") == "csv",
             csv_handler)
router.route(lambda msg: True, default_handler)   # catch-all

# Dispatch three messages with different content types
await router.dispatch(AdvancedMessage(topic="ingest.upload",
                                      content={"type": "pdf", "filename": "report.pdf"}))
await router.dispatch(AdvancedMessage(topic="ingest.upload",
                                      content={"type": "csv", "filename": "data.csv"}))
await router.dispatch(AdvancedMessage(topic="ingest.upload",
                                      content={"type": "xml", "filename": "config.xml"}))

print()
print('Routing results:')
for route_key, result in router_received.items():
    print(f'  {route_key}: {result!r}')

assert "pdf"     in router_received
assert "csv"     in router_received
assert "default" in router_received
print()
print('MessageRouter content-based routing verified.')


## Section 12 — Scatter-Gather

`bus.scatter_gather(msg, gather_count=K, timeout=T)` broadcasts a message to **all**
matching subscribers and collects their return values.

- If `gather_count=K` is set, the call returns as soon as K handlers have replied
  (or `timeout` seconds elapse, whichever is first).
- If `gather_count` is omitted, it waits for all handlers.

This is the foundation for:
- **Voting / consensus** — broadcast a proposal; collect votes from N agents.
- **Parallel search** — send the same query to N search backends; return the
  first K results.
- **Redundant computation** — run N replicas; take the majority answer.

We register 3 handlers and use `gather_count=2` so the call returns as soon as
any 2 have completed.


In [ ]:
import asyncio

scatter_bus = AdvancedMessageBus()

async def worker_a(msg: AdvancedMessage) -> dict:
    await asyncio.sleep(0.01)   # fast worker
    result = {"worker": "A", "score": 0.91, "answer": "Paris"}
    print(f'  [worker_A] done: {result}')
    return result

async def worker_b(msg: AdvancedMessage) -> dict:
    await asyncio.sleep(0.02)   # medium worker
    result = {"worker": "B", "score": 0.87, "answer": "Paris"}
    print(f'  [worker_B] done: {result}')
    return result

async def worker_c(msg: AdvancedMessage) -> dict:
    await asyncio.sleep(0.5)    # slow worker — may be cut off by gather_count
    result = {"worker": "C", "score": 0.75, "answer": "Lyon"}
    print(f'  [worker_C] done: {result}')
    return result

scatter_bus.subscribe_handler("vote.query", worker_a)
scatter_bus.subscribe_handler("vote.query", worker_b)
scatter_bus.subscribe_handler("vote.query", worker_c)

print('Scattering query to 3 workers (gather_count=2, timeout=1.0)...')

results = await scatter_bus.scatter_gather(
    AdvancedMessage(topic="vote.query", content={"question": "Capital of France?"}),
    gather_count=2,
    timeout=1.0,
)

print()
print(f'Collected {len(results)} result(s) (expected: 2 — first two fast workers):')
for r in results:
    if r is not None:
        print(f'  {r}')

assert len(results) >= 1, f"Expected at least 1 result, got {len(results)}"
print()
print('Scatter-gather verified: collected first results within timeout.')
print('Worker C (slow) may or may not have contributed depending on timing.')


## Summary

### AdvancedMessageBus API at a glance

| Method | Signature | What it does |
|--------|-----------|-------------|
| `subscribe` | `(pattern, filter_fn=None)` | Decorator: register async handler for topic pattern |
| `subscribe_handler` | `(pattern, handler, filter_fn=None)` | Programmatic subscribe; returns sub_id |
| `unsubscribe` | `(sub_id)` | Remove a subscription by ID |
| `publish` | `(msg)` | Deliver to all matching subscribers; returns delivered count |
| `request` | `(msg, timeout=10.0)` | Publish and await correlated reply |
| `scatter_gather` | `(msg, gather_count=None, timeout=5.0)` | Broadcast; collect K return values |
| `stats` | property | Dict of published / delivered / dead_lettered counts |

### Routing and filtering

| Class | Purpose |
|-------|---------|
| `MessageFilter` | Predicate builder; compose with `&`, `\|`, `~` |
| `MessagePipeline` | Async middleware chain applied before handler |
| `MessageRouter` | Content-based dispatch — first matching predicate wins |

### Priority and durability

| Feature | Behaviour |
|---------|-----------|
| `priority` field | Lower int = higher urgency; `PriorityMessageQueue` preserves order |
| `max_retries=N` | Handler is called up to N+1 times before dead-lettering |
| `DeadLetterQueue` | Captures no-subscriber and failed-handler messages; `drain()` to inspect |

### Wildcard topic syntax

| Pattern | Matches |
|---------|---------|
| `"agent.done"` | Exact match only |
| `"agent.*"` | One segment: `agent.done`, `agent.start` |
| `"agent.**"` | Any depth: `agent.x`, `agent.x.y.z` |

### Next steps

- **Notebook 26**: Session Management and Reactive State
- **Notebook 17**: Basic event bus messaging (`InMemoryBus`, `Message`)
- Combine `AdvancedMessageBus` with `Runtime` for production agent pipelines
- Use `MessagePipeline` to inject OpenTelemetry spans into every message delivery
